# EXP-004: lr 미세 튜닝 (lr=0.02) - Playground Series S6E5

**근거:** EXP-001 직후 진단에서 lr=0.05가 거의 천장이라 봤지만, EXP-003에서 CAT iter 확대로 OOF 0.94933까지 올라가며 진단이 부정확했을 가능성. lr=0.02 + n_est 충분히 확장으로 단일 모델 자연 수렴 천장 재확인.

**변경점 vs EXP-003:**
- lr 0.05 → **0.02**
- XGB n_est 2000 → **5000** + ES200, GPU
- LGB n_est 2000 → **5000** + ES200, CPU
- CAT iterations 10000 → **20000** + ES200, GPU
- 동일 split (seed=42)

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

TARGET = 'PitNextLap'
CAT_COLS = ['Driver', 'Compound', 'Race']
y = train[TARGET].astype(int).values

train_le = train.copy(); test_le = test.copy()
for c in CAT_COLS:
    le = LabelEncoder()
    train_le[c] = le.fit_transform(train_le[c].astype(str))
    seen = set(le.classes_)
    test_le[c] = test_le[c].astype(str).map(lambda v: le.transform([v])[0] if v in seen else -1)

train_cb = train.copy(); test_cb = test.copy()
for c in CAT_COLS:
    train_cb[c] = train_cb[c].astype(str)
    test_cb[c]  = test_cb[c].astype(str)

drop_cols = ['id', TARGET]
feature_cols = [c for c in train.columns if c not in drop_cols]
X_le, X_test_le = train_le[feature_cols], test_le[feature_cols]
X_cb, X_test_cb = train_cb[feature_cols], test_cb[feature_cols]

N_FOLDS, SEED = 5, 42
LR = 0.02
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(skf.split(X_le, y))
print(f'Train: {train.shape}, Test: {test.shape}, lr={LR}')

Train: (439140, 16), Test: (188165, 15), lr=0.02


## 1. XGB (GPU, lr=0.02, n_est=5000+ES200)

In [2]:
from xgboost import XGBClassifier

xgb_params = dict(
    n_estimators=5000, max_depth=8, learning_rate=LR,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
    objective='binary:logistic', eval_metric='auc',
    device='cuda', tree_method='hist',
    random_state=SEED, verbosity=0,
    early_stopping_rounds=200,
)

oof_xgb = np.zeros(len(X_le))
test_xgb = np.zeros(len(X_test_le))
xgb_iters = []
t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = XGBClassifier(**xgb_params)
    m.fit(X_le.iloc[tr_idx], y[tr_idx],
          eval_set=[(X_le.iloc[va_idx], y[va_idx])], verbose=0)
    oof_xgb[va_idx] = m.predict_proba(X_le.iloc[va_idx])[:, 1]
    test_xgb += m.predict_proba(X_test_le)[:, 1] / N_FOLDS
    xgb_iters.append(m.best_iteration)
    print(f'XGB fold {fold}: AUC={roc_auc_score(y[va_idx], oof_xgb[va_idx]):.5f}, best_iter={m.best_iteration}')

xgb_oof_auc = roc_auc_score(y, oof_xgb)
print(f'XGB OOF AUC: {xgb_oof_auc:.5f}  (best_iter mean {np.mean(xgb_iters):.0f}, elapsed {time.time()-t0:.1f}s) [GPU lr={LR}]')

XGB fold 0: AUC=0.95072, best_iter=2350
XGB fold 1: AUC=0.94855, best_iter=2232
XGB fold 2: AUC=0.94935, best_iter=1868
XGB fold 3: AUC=0.94887, best_iter=2581
XGB fold 4: AUC=0.94988, best_iter=2408
XGB OOF AUC: 0.94946  (best_iter mean 2288, elapsed 70.4s) [GPU lr=0.02]


## 2. LGB (CPU, lr=0.02, n_est=5000+ES200)

In [3]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

lgb_params = dict(
    n_estimators=5000, learning_rate=LR, num_leaves=64, max_depth=-1,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    objective='binary', metric='auc',
    random_state=SEED, n_jobs=-1, verbose=-1,
)

oof_lgb = np.zeros(len(X_le))
test_lgb = np.zeros(len(X_test_le))
lgb_iters = []
t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = LGBMClassifier(**lgb_params)
    m.fit(X_le.iloc[tr_idx], y[tr_idx],
          eval_set=[(X_le.iloc[va_idx], y[va_idx])],
          callbacks=[early_stopping(200, verbose=False), log_evaluation(0)])
    oof_lgb[va_idx] = m.predict_proba(X_le.iloc[va_idx])[:, 1]
    test_lgb += m.predict_proba(X_test_le)[:, 1] / N_FOLDS
    lgb_iters.append(m.best_iteration_)
    print(f'LGB fold {fold}: AUC={roc_auc_score(y[va_idx], oof_lgb[va_idx]):.5f}, best_iter={m.best_iteration_}')

lgb_oof_auc = roc_auc_score(y, oof_lgb)
print(f'LGB OOF AUC: {lgb_oof_auc:.5f}  (best_iter mean {np.mean(lgb_iters):.0f}, elapsed {time.time()-t0:.1f}s) [CPU lr={LR}]')

LGB fold 0: AUC=0.95053, best_iter=4364
LGB fold 1: AUC=0.94847, best_iter=2887
LGB fold 2: AUC=0.94935, best_iter=3954
LGB fold 3: AUC=0.94860, best_iter=3702
LGB fold 4: AUC=0.94972, best_iter=4727
LGB OOF AUC: 0.94933  (best_iter mean 3927, elapsed 115.1s) [CPU lr=0.02]


## 3. CatBoost (GPU, lr=0.02, iterations=20000+ES200)

In [4]:
from catboost import CatBoostClassifier

cat_idx = [feature_cols.index(c) for c in CAT_COLS]

cb_params = dict(
    iterations=20000, learning_rate=LR, depth=8, l2_leaf_reg=3.0,
    bagging_temperature=0.5, random_strength=1,
    eval_metric='AUC', loss_function='Logloss',
    cat_features=cat_idx, random_state=SEED,
    verbose=False, allow_writing_files=False,
    task_type='GPU', devices='0',
    early_stopping_rounds=200,
)

oof_cat = np.zeros(len(X_cb))
test_cat = np.zeros(len(X_test_cb))
cat_iters = []
t0 = time.time()
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = CatBoostClassifier(**cb_params)
    m.fit(X_cb.iloc[tr_idx], y[tr_idx],
          eval_set=(X_cb.iloc[va_idx], y[va_idx]),
          use_best_model=True)
    oof_cat[va_idx] = m.predict_proba(X_cb.iloc[va_idx])[:, 1]
    test_cat += m.predict_proba(X_test_cb)[:, 1] / N_FOLDS
    cat_iters.append(m.get_best_iteration())
    print(f'CAT fold {fold}: AUC={roc_auc_score(y[va_idx], oof_cat[va_idx]):.5f}, best_iter={m.get_best_iteration()}')

cat_oof_auc = roc_auc_score(y, oof_cat)
print(f'CAT OOF AUC: {cat_oof_auc:.5f}  (best_iter mean {np.mean(cat_iters):.0f}, elapsed {time.time()-t0:.1f}s) [GPU lr={LR}]')

Default metric period is 5 because AUC is/are not implemented for GPU


CAT fold 0: AUC=0.95086, best_iter=8901


Default metric period is 5 because AUC is/are not implemented for GPU


CAT fold 1: AUC=0.94858, best_iter=7867


Default metric period is 5 because AUC is/are not implemented for GPU


CAT fold 2: AUC=0.94963, best_iter=7951


Default metric period is 5 because AUC is/are not implemented for GPU


CAT fold 3: AUC=0.94911, best_iter=11257


Default metric period is 5 because AUC is/are not implemented for GPU


CAT fold 4: AUC=0.94990, best_iter=9958
CAT OOF AUC: 0.94961  (best_iter mean 9187, elapsed 2715.2s) [GPU lr=0.02]


## 4. Blending

In [5]:
oof_blend_eq  = (oof_xgb + oof_lgb + oof_cat) / 3
test_blend_eq = (test_xgb + test_lgb + test_cat) / 3
auc_eq = roc_auc_score(y, oof_blend_eq)

best = (None, -1)
for w_xgb in np.arange(0, 1.001, 0.05):
    for w_lgb in np.arange(0, 1.001 - w_xgb, 0.05):
        w_cat = 1 - w_xgb - w_lgb
        if w_cat < 0: continue
        s = w_xgb * oof_xgb + w_lgb * oof_lgb + w_cat * oof_cat
        a = roc_auc_score(y, s)
        if a > best[1]:
            best = ((round(w_xgb, 2), round(w_lgb, 2), round(w_cat, 2)), a)

w_opt, auc_opt = best
oof_blend_opt  = w_opt[0] * oof_xgb + w_opt[1] * oof_lgb + w_opt[2] * oof_cat
test_blend_opt = w_opt[0] * test_xgb + w_opt[1] * test_lgb + w_opt[2] * test_cat

print('=== EXP-004 결과 (lr=0.02) ===')
print(f'XGB         OOF AUC: {xgb_oof_auc:.5f}  [GPU]')
print(f'LGB         OOF AUC: {lgb_oof_auc:.5f}  [CPU]')
print(f'CAT         OOF AUC: {cat_oof_auc:.5f}  [GPU]')
print(f'Blend equal       :  {auc_eq:.5f}')
print(f'Blend opt {w_opt}: {auc_opt:.5f}')
print()
print('=== vs EXP-003 (lr=0.05) ===')
print(f'  XGB   0.94913 -> {xgb_oof_auc:.5f}  ({xgb_oof_auc-0.94913:+.5f})')
print(f'  LGB   0.94906 -> {lgb_oof_auc:.5f}  ({lgb_oof_auc-0.94906:+.5f})')
print(f'  CAT   0.94933 -> {cat_oof_auc:.5f}  ({cat_oof_auc-0.94933:+.5f})')
print(f'  Blend 0.95030 -> {auc_eq:.5f}      ({auc_eq-0.95030:+.5f})')

=== EXP-004 결과 (lr=0.02) ===
XGB         OOF AUC: 0.94946  [GPU]
LGB         OOF AUC: 0.94933  [CPU]
CAT         OOF AUC: 0.94961  [GPU]
Blend equal       :  0.95042
Blend opt (np.float64(0.25), np.float64(0.25), np.float64(0.5)): 0.95048

=== vs EXP-003 (lr=0.05) ===
  XGB   0.94913 -> 0.94946  (+0.00033)
  LGB   0.94906 -> 0.94933  (+0.00027)
  CAT   0.94933 -> 0.94961  (+0.00028)
  Blend 0.95030 -> 0.95042      (+0.00012)


## 5. Submissions

In [6]:
for name, prob in [
    ('exp004_xgb',         test_xgb),
    ('exp004_lgb',         test_lgb),
    ('exp004_cat',         test_cat),
    ('exp004_blend_equal', test_blend_eq),
    ('exp004_blend_opt',   test_blend_opt),
]:
    sub = pd.DataFrame({'id': test['id'], 'PitNextLap': prob})
    path = f'../submissions/submission_{name}.csv'
    sub.to_csv(path, index=False)
    print(f'  saved {path}  mean={prob.mean():.4f}')

  saved ../submissions/submission_exp004_xgb.csv  mean=0.1974
  saved ../submissions/submission_exp004_lgb.csv  mean=0.1972
  saved ../submissions/submission_exp004_cat.csv  mean=0.1979
  saved ../submissions/submission_exp004_blend_equal.csv  mean=0.1975
  saved ../submissions/submission_exp004_blend_opt.csv  mean=0.1976
